# 03F · Inference — per-sentence Left/Right/Neutral for US + BR — INFER ONLY

Loads the fine-tuned model from notebook 05 and labels the two target corpora
with the **general political stance (T4)** using the **`multi_label=False`** rule
(HF zero-shot default / Laurer's grouped decision): for each sentence, take the
**entailment logit** of each of the three production hypotheses, **softmax across
the three candidates**, and `argmax` — **no threshold, no calibration**. This is
the rule chosen in 05 (test MCC 0.748 vs 0.736 for the per-hypothesis rule).
Neutral is **retained**.

Outputs one row per sentence: `doc_id, chunk_id, stance, p_left, p_right,
p_neutral` (the `p_*` are the across-candidate probabilities, summing to 1).

**This stage does inference and nothing else.** It reads the sentence-chunk corpora
written by 00 and never touches the topic model, so it can run in parallel with the
topic stage. The topic is attached later, in `04_irt_data.py`, which joins stance to
`chunk_topics.feather` on `chunk_id`.

Separate from training so you can run inference on more capable hardware.
Resumable chunked writes + progress bars on every long step.

## 1 · Preamble — mount, seed, GPU assert

In [ ]:
# === INFER notebook ===
import os, sys, gc, json, random, re
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/Papers/transfer_learning')
except Exception:
    REPO    = Path(os.getenv('TOPIC2IRT_ROOT', r'G:/My Drive/Papers/transfer_learning/topic2irt'))
    PROJECT = REPO.parent      # local fallback
REPO = PROJECT / 'topic2irt'
assert REPO.exists(), f'REPO not found: {REPO}'

RANDOM_STATE = 14605 - 2025 - 4        # = 12576
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

import torch
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert torch.cuda.is_available(), 'Select a GPU runtime (L4/A100 + High-RAM).'
print('torch       :', torch.__version__)
print('device      :', torch.cuda.get_device_name(0))
print('CUDA mem GB :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
print('REPO        :', REPO, '| RANDOM_STATE:', RANDOM_STATE)

Mounted at /content/drive
torch       : 2.11.0+cu128
device      : NVIDIA A100-SXM4-40GB
CUDA mem GB : 42.4
REPO        : /content/drive/MyDrive/Papers/transfer_learning/topic2irt | RANDOM_STATE: 12576


## 2 · Config

In [ ]:
MODEL_DIR   = REPO / 'models/stance_nli_bgem3'   # matches 05's saved model
MAX_LENGTH  = 192

# --- Automatic batch size from GPU capacity (thumb rule, no tuning) ---
# Model class = XLM-R-large (~0.56B params), fp16 inference, seq_len <= 192.
# Each *scored sequence* costs ~35 MB at worst-case padding. We score 3 candidate
# hypotheses per sentence, reserve ~4 GB for weights + CUDA context, and give 85%
# of the remaining memory to activations. Scales to any GPU (T4/L4/A100-40/80).
_GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
_N_CAND = 3                        # Left / Right / Neutral hypotheses per sentence
_MB_SEQ = 35                       # worst-case activation per scored seq @ len 192, fp16
INFER_BATCH = max(64, int((_GPU_GB - 4) * 1000 / _MB_SEQ * 0.85 / _N_CAND) // 64 * 64)
print(f'GPU {_GPU_GB:.0f} GB -> INFER_BATCH = {INFER_BATCH} sentences '
      f'({INFER_BATCH*_N_CAND} scored seqs, ~{INFER_BATCH*_N_CAND*_MB_SEQ/1000:.0f} GB worst-case peak)')

# Smoke first: label a small random sample (writes *_SMOKE.csv). Flip to False for full.
SMOKE   = False
SMOKE_N = 2000

OUT_US = REPO / 'data/stance/stance_pred_us.csv'
OUT_BR = REPO / 'data/stance/stance_pred_br.csv'
print('model dir:', MODEL_DIR, '| SMOKE:', SMOKE)

# --- wall time of each step, appended to the pipeline timing register at the end ---
import time as _time
from datetime import datetime
RUN_STARTED = datetime.now()
TIMING = {'us': {}, 'br': {}}
SCALE  = {'us': {}, 'br': {}}
_T0 = {}
def tic():
    _T0['t'] = _time.perf_counter()
def toc(corpus, step):
    TIMING[corpus][step] = _time.perf_counter() - _T0['t']
    print(f"  [time] {corpus} {step}: {TIMING[corpus][step]:,.1f}s")


GPU 42 GB -> INFER_BATCH = 256 sentences (768 scored seqs, ~27 GB worst-case peak)
model dir: /content/drive/MyDrive/Papers/transfer_learning/topic2irt/models/stance_nli_bgem3 | SMOKE: False


## 3 · Load fine-tuned model + tokenizer + hypotheses

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
meta = json.loads((MODEL_DIR / 'stance_nli_meta.json').read_text())
LRN_ORDER = meta['lrn_order']
GEN_HYPS  = meta['general_hypotheses']
MAX_LENGTH = meta.get('max_length', MAX_LENGTH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()
print('loaded:', meta['model_name'], '| test MCC (from training):', meta.get('test_mcc'),
      '| balanced acc:', meta.get('test_balanced_acc'))
for cl, h in zip(LRN_ORDER, GEN_HYPS):
    print(f'  {cl:<8} -> "{h}"')

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

loaded: MoritzLaurer/bge-m3-zeroshot-v2.0 | test MCC (from training): 0.7478109321209481 | balanced acc: 0.8286491528162024
  Left     -> "This text expresses a leftist political stance."
  Right    -> "This text expresses a rightist political stance."
  Neutral  -> "This text expresses a neutral political stance."


## 4 · Stance scoring helper (`multi_label=False` — argmax entailment logit across candidates)

In [ ]:
ENT_IDX = model.config.label2id['entailment']

@torch.no_grad()
def score_hypotheses(texts, hyps, batch_size=128, desc='score'):
    '''Zero-shot pipeline DEFAULT (multi_label=False): raw ENTAILMENT logit per candidate,
    softmax ACROSS candidates -> P(candidate | x), shape (len(texts), len(hyps)).
    Matches HF pipeline(..., multi_label=False) and Laurer's grouped decision (argmax z_entail);
    z_not-entail is NOT used. This is the production rule chosen in 05 (test MCC 0.748 > 0.736).'''
    model.eval()
    texts = list(texts); k = len(hyps)
    out = np.zeros((len(texts), k), dtype=np.float32)
    for s in tqdm(range(0, len(texts), batch_size), desc=desc):
        b = texts[s:s+batch_size]
        prem = [t for t in b for _ in range(k)]
        hy   = [h for _ in b for h in hyps]
        enc = tokenizer(prem, hy, truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(**enc).logits
        zE = logits.float()[:, ENT_IDX].cpu().numpy().reshape(-1, k)   # raw entailment logit / candidate
        zE = zE - zE.max(1, keepdims=True)                             # numerically-stable softmax ...
        e = np.exp(zE)
        out[s:s+len(b)] = e / e.sum(1, keepdims=True)                  # ... ACROSS the k candidates
    return out

def predict_general(texts, batch_size=128, desc='stance'):
    '''argmax over the 3 general-stance hypotheses -> Left/Right/Neutral + probs (multi_label=False).'''
    probs = score_hypotheses(texts, GEN_HYPS, batch_size=batch_size, desc=desc)
    pred = np.array(LRN_ORDER)[probs.argmax(1)]
    return pred, probs

## 5 · Resumable corpus-inference helper + mojibake repair

In [ ]:
def run_corpus_inference(df, out_csv, name, chunk_rows=200_000, resume=True):
    out_csv = Path(out_csv)
    cols = ['doc_id','chunk_id','stance','p_left','p_right','p_neutral']
    start = 0
    if resume and out_csv.exists():
        start = max(0, sum(1 for _ in open(out_csv, encoding='utf-8')) - 1)
        print(f'[{name}] resuming after {start:,} rows')
    n = len(df)
    if start >= n:
        print(f'[{name}] already complete ({n:,} rows)'); return
    for c0 in range(start, n, chunk_rows):
        c1 = min(c0 + chunk_rows, n)
        sub = df.iloc[c0:c1]
        # Length-bucketed batching: sort each chunk by text length so every mini-batch
        # pads to a short common length instead of occasionally to MAX_LENGTH, which cuts
        # wasted padding compute (the real speed lever -- GPU here is compute-bound, not
        # memory-bound). Chunk membership stays tied to the original rows c0:c1, so
        # resume-by-row-count is unaffected and each sentence is scored exactly once;
        # only the intra-chunk order changes, and every row carries its own doc_id/chunk_id.
        order = np.argsort(sub['chunk_text'].str.len().values, kind='stable')
        sub = sub.iloc[order]
        pred, probs = predict_general(sub['chunk_text'].tolist(), batch_size=INFER_BATCH,
                                      desc=f'{name} {c0:,}-{c1:,}')
        pd.DataFrame({'doc_id': sub['doc_id'].values, 'chunk_id': sub['chunk_id'].values,
                      'stance': pred,
                      'p_left': probs[:,0], 'p_right': probs[:,1], 'p_neutral': probs[:,2]})[cols] \
            .to_csv(out_csv, mode=('w' if c0 == 0 else 'a'), header=(c0 == 0), index=False, encoding='utf-8')
    print(f'[{name}] done -> {out_csv} ({n:,} rows)')

## 6 · US corpus (CampaignView House platforms, English)
Text from `campaignview_chunks_sent.feather`. Inference only — no topic model is read.

In [ ]:
tic()
us = pd.read_feather(REPO / 'data/us/campaignview_chunks_sent.feather')[['chunk_id','doc_id','chunk_text']].copy()
us['chunk_text'] = us['chunk_text'].astype(str)
SCALE['us'] = {'n_docs': int(us['doc_id'].nunique()), 'n_chunks': int(len(us))}
print('US sentences:', f'{len(us):,}', 'over', f"{us['doc_id'].nunique():,} platforms")
toc('us', '1 load')

tic()
us_run = us.sample(SMOKE_N, random_state=RANDOM_STATE).reset_index(drop=True) if SMOKE else us
run_corpus_inference(us_run, OUT_US.with_name('stance_pred_us_SMOKE.csv') if SMOKE else OUT_US, name='US')
toc('us', '2 infer')

US sentences: 439,282 over 4,507 platforms
  [time] us 1 load: 3.6s


US 0-200,000:   0%|          | 0/782 [00:00<?, ?it/s]

US 200,000-400,000:   0%|          | 0/782 [00:00<?, ?it/s]

US 400,000-439,282:   0%|          | 0/154 [00:00<?, ?it/s]

[US] done -> /content/drive/MyDrive/Papers/transfer_learning/topic2irt/data/stance/stance_pred_us.csv (439,282 rows)
  [time] us 2 infer: 406.0s


## 7 · BR corpus (Brazilian mayoral platforms, Portuguese)
Text from `br_manifestos_chunks_sent.feather`, restricted to `valid_mayor_platform` so it
covers exactly the chunks the topic stage keeps. Inference only — no topic model is read.

In [ ]:
tic()
br = pd.read_feather(REPO / 'data/br/br_manifestos_chunks_sent.feather')[['chunk_id','doc_id','chunk_text']].copy()
# Same mayoral restriction the topic stage applies, so both cover the same chunks.
_pm = pd.read_feather(REPO / 'data/br/platform_party_map.feather',
                      columns=['platform_id', 'valid_mayor_platform'])
_mayor = set(_pm.loc[_pm['valid_mayor_platform'] == True, 'platform_id'].astype(str))
br = br[br['doc_id'].astype(str).isin(_mayor)].reset_index(drop=True)
br['chunk_text'] = br['chunk_text'].astype(str)   # corpus is clean UTF-8 -- use text verbatim
SCALE['br'] = {'n_docs': int(br['doc_id'].nunique()), 'n_chunks': int(len(br))}
print('BR sentences:', f'{len(br):,}', 'over', f"{br['doc_id'].nunique():,} mayoral platforms")
toc('br', '1 load+mayoral filter')

tic()
br_run = br.sample(SMOKE_N, random_state=RANDOM_STATE).reset_index(drop=True) if SMOKE else br
run_corpus_inference(br_run, OUT_BR.with_name('stance_pred_br_SMOKE.csv') if SMOKE else OUT_BR, name='BR')
toc('br', '2 infer')

BR sentences: 3,081,612 over 16,830 mayoral platforms
  [time] br 1 load+mayoral filter: 13.1s


BR 0-200,000:   0%|          | 0/782 [00:00<?, ?it/s]

BR 200,000-400,000:   0%|          | 0/782 [00:00<?, ?it/s]

BR 400,000-600,000:   0%|          | 0/782 [00:00<?, ?it/s]

BR 600,000-800,000:   0%|          | 0/782 [00:00<?, ?it/s]

...

BR 2,400,000-2,600,000:   0%|          | 0/782 [00:00<?, ?it/s]

BR 2,600,000-2,800,000:   0%|          | 0/782 [00:00<?, ?it/s]

BR 2,800,000-3,000,000:   0%|          | 0/782 [00:00<?, ?it/s]

BR 3,000,000-3,081,612:   0%|          | 0/319 [00:00<?, ?it/s]

[BR] done -> /content/drive/MyDrive/Papers/transfer_learning/topic2irt/data/stance/stance_pred_br.csv (3,081,612 rows)
  [time] br 2 infer: 3,475.5s


## 8 · Output sanity check

In [ ]:
for nm, full in [('US', OUT_US), ('BR', OUT_BR)]:
    p = full.with_name(full.stem + '_SMOKE.csv') if SMOKE else full
    if not Path(p).exists():
        print(f'{nm}: {p.name} not written'); continue
    d = pd.read_csv(p)
    print(f'\n== {nm}: {p.name} ({len(d):,} rows) ==')
    print('stance dist:', d['stance'].value_counts(normalize=True).round(3).to_dict())
    print(d.head(4).to_string())


== US: stance_pred_us.csv (439,282 rows) ==
stance dist: {'Left': 0.425, 'Neutral': 0.314, 'Right': 0.26}
                                        doc_id                                         chunk_id   stance    p_left  p_right  p_neutral
0                  Ami Bera|CA|6|Democrat|2022                  Ami Bera|CA|6|Democrat|2022#s99  Neutral  0.001431  0.00038   0.998189
1            Andrew Clyde|GA|9|Republican|2022            Andrew Clyde|GA|9|Republican|2022#s25  Neutral  0.001431  0.00038   0.998189
2            Andrew Clyde|GA|9|Republican|2022            Andrew Clyde|GA|9|Republican|2022#s31  Neutral  0.001431  0.00038   0.998189
3  Ane Roseborough-Eberhard|NJ|8|Democrat|2022  Ane Roseborough-Eberhard|NJ|8|Democrat|2022#s16  Neutral  0.001431  0.00038   0.998189

== BR: stance_pred_br.csv (3,081,612 rows) ==
stance dist: {'Neutral': 0.487, 'Left': 0.446, 'Right': 0.067}
              doc_id                chunk_id   stance    p_left   p_right  p_neutral
0  2020AC10000733704   

In [ ]:
# --- append this run to the pipeline timing register (code/timing.py is its only writer) ---
import sys
sys.path.insert(0, str(REPO / 'code'))
from timing import log_run, CSV_PATH, MD_PATH

note = (f"{torch.cuda.get_device_name(0)}, batch {INFER_BATCH}, max_length {MAX_LENGTH}, fp16. "
        f"Inference only: no topic model is read.")
print(log_run('03F stance inference', {c: t for c, t in TIMING.items() if t},
              scale={c: SCALE[c] for c in TIMING if TIMING[c]},
              started=RUN_STARTED, note=note))
print('->', CSV_PATH)
print('->', MD_PATH)

## 03F stance inference · 2026-08-13 03:13:51 · 09323a9093e6, 12 cores

NVIDIA A100-SXM4-40GB, batch 256, max_length 192, fp16. Inference only: no topic model is read.

| step | United States | Brazil |
|---|---:|---:|
| 1 load | 0:00:04 | 0:00:00 |
| 2 infer | 0:06:46 | 0:57:55 |
| 1 load+mayoral filter | 0:00:00 | 0:00:13 |
| **total** | **0:06:50** | **0:58:09** |

| corpus | documents | chunks | seconds per 1k chunks |
|---|---:|---:|---:|
| United States | 4,507 | 439,282 | 0.9 |
| Brazil | 16,830 | 3,081,612 | 1.1 |

Wall clock over the whole stage: **1:04:58**.

-> /content/drive/MyDrive/Papers/transfer_learning/topic2irt/reports/timing/pipeline_timing.csv
-> /content/drive/MyDrive/Papers/transfer_learning/topic2irt/reports/timing/pipeline_timing.md


In [ ]:
# --- 9 · Auto-disconnect: free the GPU when inference finishes (avoid idle credit burn) ---
# runtime.unassign() terminates this Colab runtime and releases the GPU (equivalent to
# "Disconnect and delete runtime"). It KILLS the kernel -- in-memory state is lost and the
# colab-proxy link drops -- but every output is already persisted to Drive by
# run_corpus_inference (resumable CSVs), so it is safe as the LAST cell.
#
# Default: only auto-disconnect after a FULL run (SMOKE=False); during a smoke run the runtime
# is kept alive so you can inspect. Override by setting TERMINATE explicitly either way.
TERMINATE = (not SMOKE)
if TERMINATE:
    from google.colab import runtime
    print('Inference finished (full run) -- disconnecting runtime and releasing the GPU now.')
    runtime.unassign()
else:
    print('SMOKE run -- runtime kept alive for inspection (set TERMINATE=True to force disconnect).')

Inference finished (full run) -- disconnecting runtime and releasing the GPU now.
